In [1]:
import os
import sys


os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data Aggregation") \
    .getOrCreate()

In [2]:
listings = spark.read.csv("data/listings.csv.gz",
                          header= True,
                          inferSchema=True,
                          sep=",", #SEPARATOR
                          quote='"', #pentru valorile string
                          escape='"',
                          multiLine=True, 
                          mode="PERMISSIVE"
                         )

In [15]:
listings.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [3]:
reviews = spark.read.csv("data/reviews.csv.gz",
                          header= True,
                          inferSchema=True,
                          sep=",", #SEPARATOR
                          quote='"', #pentru valorile string
                          escape='"',
                          multiLine=True, 
                          mode="PERMISSIVE"
                         )

In [13]:
reviews.printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)



In [5]:
listing_reviews = listings.join(
    reviews, listings.id == reviews.listing_id, how = 'inner')


In [16]:
#import pyspark.sql.functions as F 
#reviews_per_listing = listing_reviews \
 #  .agg(
  #      F.count(reviews.id).alias('num_reviews')
   # )\
    #.orderBy('num_reviews', ascending=[False])\
    #.show(truncate=False)

reviews_per_listing = reviews \
  .groupBy('listing_id') \
  .count() \
  .show(10)
        

+----------+-----+
|listing_id|count|
+----------+-----+
|     78606|    2|
|    444886|   12|
|    466017|   28|
|    991477|    5|
|   2557853|   92|
|   2736493|    4|
|   3132302|    3|
|   3734796|    5|
|   3997029|    7|
|   3917692|    1|
+----------+-----+
only showing top 10 rows


In [17]:
from pyspark.sql.functions import avg, count
total_number_and_avg = listings \
    .filter(listings.review_scores_rating.isNotNull()) \
    .groupby('host_id') \
    .agg(
        count('id').alias('total_listigs'),
        avg('review_scores_rating').alias('average_review_score')
    ) \
    .show(10)

+--------+-------------+--------------------+
| host_id|total_listigs|average_review_score|
+--------+-------------+--------------------+
| 2038199|            1|                 5.0|
| 2358441|            1|                4.86|
| 2876123|            2|  4.9399999999999995|
| 4157822|            2|  4.9350000000000005|
|  719504|            1|                4.96|
| 7950720|            1|                4.86|
| 6572018|            1|                 5.0|
|12122942|            1|                4.93|
|13851928|            1|                4.97|
|13739634|            2|                4.75|
+--------+-------------+--------------------+
only showing top 10 rows


In [20]:
reviews \
    .groupby('listing_id') \
    .count() \
    .orderBy('count', ascending = [False]) \
    .limit(10) \
    .show()

+----------+-----+
|listing_id|count|
+----------+-----+
|  47408549| 1902|
|  43120947| 1647|
|  19670926| 1443|
|   2126708| 1142|
|  46233904| 1002|
|   2659707|  998|
|  27833488|  951|
|   4748665|  933|
|  42081759|  914|
|   5266466|  909|
+----------+-----+



In [22]:
listings \
    .groupby('neighbourhood_cleansed') \
    .count() \
    .orderBy('count', ascending=False) \
    .show(5, truncate=False)

+----------------------+-----+
|neighbourhood_cleansed|count|
+----------------------+-----+
|Westminster           |11385|
|Tower Hamlets         |7469 |
|Camden                |6551 |
|Kensington and Chelsea|6401 |
|Hackney               |6359 |
+----------------------+-----+
only showing top 5 rows


In [25]:
listing_and_review = listing_reviews \
    .select(listings.id, listings.name, reviews.reviewer_name, reviews.comments) \
    .show()

+-----+--------------------+-------------+--------------------+
|   id|                name|reviewer_name|            comments|
+-----+--------------------+-------------+--------------------+
|13913|Holiday London DB...|      Michael|My girlfriend and...|
|13913|Holiday London DB...|      Mathias|Alina was a reall...|
|13913|Holiday London DB...|      Kristin|Alina is an amazi...|
|13913|Holiday London DB...|      Camilla|Alina's place is ...|
|13913|Holiday London DB...|        Jorik|Nice location in ...|
|13913|Holiday London DB...|         Vera|I'm very happy to...|
|13913|Holiday London DB...|         Honi|I stayed with Ali...|
|13913|Holiday London DB...|   Alessandro|Alina was a perfe...|
|13913|Holiday London DB...|         Oleh|Alina's flat is e...|
|13913|Holiday London DB...|           Mo|The House is a pi...|
|13913|Holiday London DB...|            A|Was great base fo...|
|13913|Holiday London DB...|       Daniel|Alina was an amaz...|
|13913|Holiday London DB...|      Belind

In [29]:
from pyspark.sql.functions import length, avg, count

comm_length_review = reviews.withColumn('comm_length',  length('comments')) 
comm_length_review \
    .join( listings, comm_length_review.listing_id == listings.id, 'inner') \
    .groupby('listing_id').agg(
        avg(comm_length_review.comm_length).alias('average_comment_length'),
        count(comm_length_review.id).alias('reviews_count')
    )\
    .filter('reviews_count >= 5') \
    .orderBy('average_comment_length', ascending = False) \
    .show()

+------------------+----------------------+-------------+
|        listing_id|average_comment_length|reviews_count|
+------------------+----------------------+-------------+
|618608352812465378|    1300.1666666666667|            6|
|          28508447|    1089.3333333333333|            6|
|          22661311|     1035.857142857143|            7|
|          53145228|    1006.6666666666666|            6|
|627425975703032358|     951.7777777777778|            9|
|           2197681|                 939.2|            5|
|          13891813|                 905.0|            5|
|            979753|     893.9230769230769|           13|
|630150178279666225|     890.7272727272727|           11|
|           8856894|     890.1666666666666|            6|
|          33310686|     885.8333333333334|            6|
|          22524075|                 885.0|            5|
|          29469389|                 885.0|            6|
|           5555679|     878.7169811320755|          106|
|           65

In [33]:
lonely_listings = listings \
    .join(reviews, listings.id == reviews.listing_id, 'left_anti') \
    .select('name') \
    .show(truncate = False)

+-------------------------------------------------+
|name                                             |
+-------------------------------------------------+
|ChiqDoube Room in PrivateAppartment              |
|ROOM TO RENT IN THE OLYMPIC PERIOD               |
|London, Hoxton. Nice, 2 bedroom, 7th floor flat. |
|Luxury Central London House with Gym and Reformer|
|Bright Dbl/Nr/ Excellnt Transp                   |
|Stunning Shared Penthouse Apartment              |
|The Old Coach House (Olympics)                   |
|Studio 20min Walk from Olympic City              |
|Luxury single room                               |
|Contemporary house London E4                     |
|A lovely one bedroom garden flat!!               |
|Coming to London for the Olympics?               |
|Double Room close to Olympic Park!               |
|Double bedroom near Olympic Park                 |
|SPARE ROOM TO LET DURING OLYMPICS                |
|Your Studio Flat in London-Olympics              |
|1 bedroom a